# Minimum information tiers: gene clusters and co-expression

Which genes are putatively in the same pathway because they lie in the same biosynthetic gene cluster (lowest tier), are also co-expressed (here: in the same MEANtools functional cluster), and match PlantMetWiki pathways?

**Case:** cases 3 and 4: plantiSMASH-predicted BGCs and gene co-expression (with case 6). **Data:** the association graph, the sample graph (the Arabidopsis cluster) and the **tomato cluster graph** `pilots/plantismash_overlap/results/bgc_tomato.ttl` (built by `build_tomato_bgc.py`: the genes of the 63 tomato plantiSMASH clusters that map to a Solyc locus, about half of the cluster genes). **Queries:** the files of [`sparql-examples-pmw`](https://github.com/pathway-lod/sparql-examples-pmw/tree/main/examples/MinimumInformationTiers) (<https://pathway-lod.github.io/sparql-examples-pmw/>), run in a local Oxigraph store, with PlantMetWiki, Rhea and UniProt called by `SERVICE` on the live endpoints.

The outputs are a **dated snapshot**: the date and the PlantMetWiki release are in the summary table at the end. Each query is run once with a time limit; a tier that did not finish is shown as *timeout*. The results of every tier are also saved as CSV in `results/`. To rerun, open the notebook with a kernel that has the packages of `requirements.txt` installed (see the README of this folder).

**What changed (2026-11-18).** The first version had no tomato clusters in the graphs, and the three co-expression tiers used a join (`STRENDS` of every cluster gene against every association gene) that ran out of time. The tomato clusters are now in the graph with the same gene identifiers as the associations, the join is exact, and the tiers finish in seconds. Their remaining zero rows are a **computed result**, explained below.

In [1]:
import mi
import pandas as pd
from IPython.display import display
pd.set_option('display.max_colwidth', 80)
CASE = '3 gene clusters'
print('PlantMetWiki release:', mi.pmw_release())

PlantMetWiki release: r17.0.0_20260605-171052


## The data

In [2]:
data = ['association', 'sample', 'bgc_tomato']
print('gene-metabolite associations:', mi.local_count(data, 'PREFIX lodp: <https://example.org/lod-pathways/pilot/> SELECT (COUNT(?a) AS ?n) WHERE { ?a a lodp:GeneMetaboliteAssociation }'))
print('MEANtools functional clusters:', mi.local_count(data, 'PREFIX lodp: <https://example.org/lod-pathways/pilot/> PREFIX sio: <http://semanticscience.org/resource/> SELECT (COUNT(DISTINCT ?c) AS ?n) WHERE { ?a a lodp:GeneMetaboliteAssociation ; sio:SIO_000772/lodp:functionalCluster ?c }'))
print('biosynthetic gene clusters in the sample graph:', mi.local_count(data, 'PREFIX obo: <http://purl.obolibrary.org/obo/> SELECT (COUNT(DISTINCT ?c) AS ?n) WHERE { ?c a obo:SO_0002357 }'))
Q = 'PREFIX lodp: <https://example.org/lod-pathways/pilot/> PREFIX sio: <http://semanticscience.org/resource/> PREFIX obo: <http://purl.obolibrary.org/obo/> '
print('genes of the associations that are in a tomato biosynthetic gene cluster:', mi.local_count(data, Q + 'SELECT (COUNT(DISTINCT ?g) AS ?n) WHERE { ?a a lodp:GeneMetaboliteAssociation ; sio:SIO_000628 ?g . ?c a obo:SO_0002357 ; obo:BFO_0000051 ?g }'))
print('clusters that contain two or more genes of the associations:', mi.local_count(data, Q + 'SELECT (COUNT(DISTINCT ?c) AS ?n) WHERE { ?a1 a lodp:GeneMetaboliteAssociation ; sio:SIO_000628 ?g1 . ?a2 a lodp:GeneMetaboliteAssociation ; sio:SIO_000628 ?g2 . FILTER(?g1 != ?g2) ?c a obo:SO_0002357 ; obo:BFO_0000051 ?g1 , ?g2 }'))

gene-metabolite associations: 1294
MEANtools functional clusters: 53


biosynthetic gene clusters in the sample graph: 64


genes of the associations that are in a tomato biosynthetic gene cluster: 7
clusters that contain two or more genes of the associations: 0


## Genes in the same biosynthetic gene cluster (lowest tier)

`genes-in-same-bgc`

genes-in-same-bgc: Two or more genes lie in the same biosynthetic gene cluster (cluster has part gene). The lowest tier for genes of the same pathway. OUTPUT: a list of genes (the lowest tier: putatively in the same pathway because two or more genes lie in the same biosynthetic gene cluster), with the cluster and the number of its genes as metadata. No PlantMetWiki and no co-expression data are needed.

*Keywords:* Minimum Information: evidence tiers, biosynthetic gene cluster, case 3: plantiSMASH-predicted BGCs, predicted gene clusters, evidence tier, gene, local evidence graph, output: genes, plantiSMASH

In [3]:
r = mi.run('genes-in-same-bgc', ['sample', 'bgc_tomato'], CASE, timeout=150)
mi.record(r)
print(r['status'], '|', r['n_rows'], 'rows |', r['seconds'], 's')
display(r['table'].head(10))

rows | 506 rows | 0.0 s


,component,componentType,cluster,clusterSize
0,https://plants.ensembl.org/Solanum_lycopersicum/Gene/Summary?g=Solyc03g114050,gene,https://plantismash.bioinformatics.nl/precalc/v2/Solanum_lycopersicum_GCF_03...,28
1,https://plants.ensembl.org/Solanum_lycopersicum/Gene/Summary?g=Solyc03g114060,gene,https://plantismash.bioinformatics.nl/precalc/v2/Solanum_lycopersicum_GCF_03...,28
2,https://plants.ensembl.org/Solanum_lycopersicum/Gene/Summary?g=Solyc03g114070,gene,https://plantismash.bioinformatics.nl/precalc/v2/Solanum_lycopersicum_GCF_03...,28
3,https://plants.ensembl.org/Solanum_lycopersicum/Gene/Summary?g=Solyc03g114100,gene,https://plantismash.bioinformatics.nl/precalc/v2/Solanum_lycopersicum_GCF_03...,28
4,https://plants.ensembl.org/Solanum_lycopersicum/Gene/Summary?g=Solyc03g114120,gene,https://plantismash.bioinformatics.nl/precalc/v2/Solanum_lycopersicum_GCF_03...,28
5,https://plants.ensembl.org/Solanum_lycopersicum/Gene/Summary?g=Solyc03g114130,gene,https://plantismash.bioinformatics.nl/precalc/v2/Solanum_lycopersicum_GCF_03...,28
6,https://plants.ensembl.org/Solanum_lycopersicum/Gene/Summary?g=Solyc03g114140,gene,https://plantismash.bioinformatics.nl/precalc/v2/Solanum_lycopersicum_GCF_03...,28
7,https://plants.ensembl.org/Solanum_lycopersicum/Gene/Summary?g=Solyc03g114150,gene,https://plantismash.bioinformatics.nl/precalc/v2/Solanum_lycopersicum_GCF_03...,28
8,https://plants.ensembl.org/Solanum_lycopersicum/Gene/Summary?g=Solyc03g114160,gene,https://plantismash.bioinformatics.nl/precalc/v2/Solanum_lycopersicum_GCF_03...,28
9,https://plants.ensembl.org/Solanum_lycopersicum/Gene/Summary?g=Solyc03g114180,gene,https://plantismash.bioinformatics.nl/precalc/v2/Solanum_lycopersicum_GCF_03...,28


**Reading.** 506 gene-cluster rows (the 63 tomato clusters and the Arabidopsis cluster of the sample graph): any gene of a cluster with two or more genes is a putative member of the cluster's pathway, the lowest tier (weight 0.30 in the ranked query).

**For discussion:** is this the right rule for the tier? Is the number of results what we expect? What would make it more useful (a threshold, another data source, a different match)?

*Comments:* 

## Co-expressed genes are in the same biosynthetic gene cluster

`coexpressed-in-same-cluster`

coexpressed-in-same-cluster: Two genes that are co-expressed (here: genes associated with the same metabolite feature in the same MEANtools functional cluster) are both parts of the same plantiSMASH biosynthetic gene cluster (cluster obo:BFO_0000051 gene). Returns the pairs and the cluster. Needs the tomato cluster graph of the plantiSMASH pilot (`bgc_tomato.ttl`: the genes of the 63 tomato clusters that map to a Solyc locus, about half of the cluster genes). On the MEANtools pilot only 6 of the 350 genes are in a tomato cluster, so it finds few or no pairs. OUTPUT: a list of genes (putatively in the same pathway: co-expressed genes that lie in the same biosynthetic gene cluster), with the cluster, the partner gene and the functional cluster of the co-expression as metadata.

*Keywords:* MEANtools, Minimum Information: evidence tiers, biosynthetic gene cluster, case 3: plantiSMASH-predicted BGCs, predicted gene clusters, case 4: transcriptomics, gene co-expression values, case 6: paired omics, transcript-metabolite associations, co-expression, evidence tier, gene, gene-metabolite association, local evidence graph, output: genes, plantiSMASH

In [4]:
r = mi.run('coexpressed-in-same-cluster', ['association', 'bgc_tomato'], CASE, timeout=600)
mi.record(r)
print(r['status'], '|', r['n_rows'], 'rows |', r['seconds'], 's')
display(r['table'].head(10))

no rows | 0 rows | 2.2 s


,component,componentType,cluster,partnerGene,functionalCluster


**Reading.** No rows, a **computed** result. Only 6 of the 350 genes with a MEANtools association are in a tomato cluster: the clusters are mostly formed by genes that are **not** part of the MEANtools functional clusters (a different set of genes and a different assembly; only about half of the cluster genes map to a Solyc locus). This says that the two kinds of evidence come from **non-overlapping gene sets** here, not that the genes are unrelated.

**For discussion:** is this the right rule for the tier? Is the number of results what we expect? What would make it more useful (a threshold, another data source, a different match)?

*Comments:* 

## Same cluster, only one of the genes matches a pathway

`coexpressed-in-cluster-one-in-pathway`

coexpressed-in-cluster-one-in-pathway: As in `coexpressed-in-same-cluster`, and only one of the two genes is listed in a PlantMetWiki pathway (by its Solyc locus). This is a lead to extend that pathway with the other gene. Runs on the association graph and the tomato cluster graph (see `coexpressed-in-same-cluster`); on the MEANtools pilot it finds no pair, because no two of the genes of the associations are in the same tomato cluster. OUTPUT: a list of genes (putative members of a pathway: a gene that is not listed in PlantMetWiki but is co-expressed with, and in the same biosynthetic gene cluster as, a gene that is in the pathway); the pathway, the gene it is inferred from and the cluster are metadata.

*Keywords:* Minimum Information: evidence tiers, PlantMetWiki pathway, Solyc locus, biosynthetic gene cluster, case 3: plantiSMASH-predicted BGCs, predicted gene clusters, case 4: transcriptomics, gene co-expression values, case 6: paired omics, transcript-metabolite associations, co-expression, evidence tier, federated query, gene, local evidence graph, output: genes, pathway expansion, plantiSMASH

In [5]:
r = mi.run('coexpressed-in-cluster-one-in-pathway', ['association', 'bgc_tomato'], CASE, timeout=600)
mi.record(r)
print(r['status'], '|', r['n_rows'], 'rows |', r['seconds'], 's')
display(r['table'].head(10))

no rows | 0 rows | 4.2 s


,component,componentType,pathway,viaGene,cluster


**Reading.** No rows, a **computed** result (before: timeout). The tier needs two genes of the associations in the same cluster, and there are none (see the data cell: 6 genes in tomato clusters, no cluster with two of them). The tier will have rows for data in which the cluster genes and the co-expressed genes overlap more, for example transcriptomic data of the genome that plantiSMASH analysed, or the Arabidopsis clusters with an Arabidopsis co-expression set.

**For discussion:** is this the right rule for the tier? Is the number of results what we expect? What would make it more useful (a threshold, another data source, a different match)?

*Comments:* 

## Same cluster, both genes match a pathway

`coexpressed-in-cluster-both-in-pathway`

coexpressed-in-cluster-both-in-pathway: As in `coexpressed-in-same-cluster`, and both genes are listed in a PlantMetWiki pathway (by their Solyc loci); returns the pathways of the two genes and whether it is the same pathway. Runs on the association graph and the tomato cluster graph (see `coexpressed-in-same-cluster`); on the MEANtools pilot it finds no pair, because no two of the genes of the associations are in the same tomato cluster. OUTPUT: a list of genes (supported members of a pathway: co-expressed genes in the same biosynthetic gene cluster that are both listed in PlantMetWiki pathways); the pathway of the gene, the partner gene and its pathway, and whether they are the same pathway are metadata.

*Keywords:* Minimum Information: evidence tiers, PlantMetWiki pathway, Solyc locus, biosynthetic gene cluster, case 3: plantiSMASH-predicted BGCs, predicted gene clusters, case 4: transcriptomics, gene co-expression values, case 6: paired omics, transcript-metabolite associations, co-expression, evidence tier, federated query, gene, local evidence graph, output: genes, plantiSMASH

In [6]:
r = mi.run('coexpressed-in-cluster-both-in-pathway', ['association', 'bgc_tomato'], CASE, timeout=600)
mi.record(r)
print(r['status'], '|', r['n_rows'], 'rows |', r['seconds'], 's')
display(r['table'].head(10))

no rows | 0 rows | 5.5 s


,component,componentType,pathway,partnerGene,partnerPathway,samePathway,cluster


**Reading.** No rows, a **computed** result (the query now finishes in seconds). Both genes would have to be in PlantMetWiki pathways (by locus, the weakest data of the three: about 30 genes) and in the same cluster; no two of the genes of the associations share a tomato cluster.

**For discussion:** is this the right rule for the tier? Is the number of results what we expect? What would make it more useful (a threshold, another data source, a different match)?

*Comments:* 

## Summary of this case

In [7]:
s = pd.read_csv('results/summary.csv')
display(s[s['case'] == CASE].reset_index(drop=True))

,tier,case,status,n_rows,seconds,timeout_s,date,pmw_release
0,coexpressed-in-cluster-both-in-pathway,3 gene clusters,no rows,0.0,5.5,600,2026-11-18,r17.0.0_20260605-171052
1,coexpressed-in-cluster-one-in-pathway,3 gene clusters,no rows,0.0,4.2,600,2026-11-18,r17.0.0_20260605-171052
2,coexpressed-in-same-cluster,3 gene clusters,no rows,0.0,2.2,600,2026-11-18,r17.0.0_20260605-171052
3,genes-in-same-bgc,3 gene clusters,rows,506.0,0.0,150,2026-11-18,r17.0.0_20260605-171052
